# NER Analysis of Equity-Related Public Health Discourse

This notebook studies which named entities appear in equity-related public health text across four project document types: academic, policy, NGO, and news.

The analysis focuses on the target terms `equity`, `equitable`, `inequity`, and `inequitable`, and pays particular attention to the entity labels `ORG`, `GPE`, and `PERSON`.

**Analytical note.** The sentence-level corpus uses source categories such as `federal_policy`, `state_local`, `ngo_nonprofit`, and `news_commentary`. To match the project framing, this notebook collapses those source categories into the broader document types used in the paper:

- `academic` -> `academic`
- `federal_policy` and `state_local` -> `policy`
- `ngo_nonprofit` -> `NGO`
- `news_commentary` -> `news`

Because the corpus is imbalanced across document types, the notebook reports both raw counts and normalized rates such as entity mentions per 1,000 equity-related contexts.

## 1. Setup

This section imports the analysis libraries, defines reusable helper functions, and sets the core notebook parameters.

In [1]:
# We import libraries and define global constants for the NER workflow.
from pathlib import Path
from collections import Counter
import re
import warnings

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")
sns.set_theme(style="whitegrid", palette="colorblind")

TARGET_TERMS = ["equity", "equitable", "inequity", "inequitable"]
FOCUS_LABELS = ["ORG", "GPE", "PERSON"]
DOC_TYPE_ORDER = ["academic", "policy", "NGO", "news"]
DOC_TYPE_COLORS = {
    "academic": "#1f4e79",
    "policy": "#e07a1f",
    "NGO": "#2a9d8f",
    "news": "#c44536",
}

print("Library versions loaded successfully:")
print(f"- pandas: {pd.__version__}")
print(f"- spaCy: {spacy.__version__}")
print(f"- seaborn: {sns.__version__}")
print(f"- matplotlib: {plt.matplotlib.__version__}")
print(f"Target terms: {', '.join(TARGET_TERMS)}")
print(f"Focus labels: {', '.join(FOCUS_LABELS)}")

Library versions loaded successfully:
- pandas: 3.0.2
- spaCy: 3.8.14
- seaborn: 0.13.2
- matplotlib: 3.10.8
Target terms: equity, equitable, inequity, inequitable
Focus labels: ORG, GPE, PERSON


In [2]:
# We define utility helpers for path detection, column selection, and document-type mapping.
def find_project_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "01_data").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the project root that contains the 01_data folder."
    )


def detect_column(columns, candidates, required=True):
    column_lookup = {column.lower(): column for column in columns}
    for candidate in candidates:
        if candidate.lower() in column_lookup:
            return column_lookup[candidate.lower()]
    if required:
        raise KeyError(f"Missing required column. Expected one of: {candidates}")
    return None


def map_project_doc_type(value):
    text = "" if pd.isna(value) else str(value).strip().lower()
    if text == "academic":
        return "academic"
    if (
        text in {"federal_policy", "state_local", "policy", "policy_report"}
        or "policy" in text
    ):
        return "policy"
    if (
        text in {"ngo_nonprofit", "ngo", "nonprofit"}
        or "ngo" in text
        or "nonprofit" in text
    ):
        return "NGO"
    if text in {"news_commentary", "news", "commentary"} or "news" in text:
        return "news"
    return "other"


def load_spacy_model():
    for model_name in ["en_core_web_lg", "en_core_web_md", "en_core_web_sm"]:
        try:
            return spacy.load(model_name), model_name
        except OSError:
            continue
    raise OSError(
        "No spaCy English model was found. Install one with: python -m spacy download en_core_web_sm"
    )


def extract_first_keyword(text):
    lowered = str(text).lower()
    for term in TARGET_TERMS:
        if term in lowered:
            return term
    return None

In [3]:
# We define the entity-extraction helper that builds a long-format NER dataset.
def build_entity_dataset(frame, text_col, doc_id_col, raw_doc_type_col, nlp):
    if frame.empty:
        return pd.DataFrame()

    metadata_columns = [
        doc_id_col,
        raw_doc_type_col,
        "project_document_type",
        "sentence_id",
        "title",
        "source_label",
        "url",
    ]
    metadata_columns = [
        column for column in metadata_columns if column in frame.columns
    ]

    records = []
    text_stream = frame[text_col].fillna("").astype(str)
    row_stream = frame[metadata_columns + [text_col]].to_dict("records")

    for row, doc in zip(row_stream, nlp.pipe(text_stream, batch_size=64)):
        for entity in doc.ents:
            entity_text = entity.text.strip()
            if len(entity_text) <= 1 or not re.search(r"[A-Za-z]", entity_text):
                continue
            records.append(
                {
                    doc_id_col: row.get(doc_id_col),
                    "sentence_id": row.get("sentence_id"),
                    "raw_document_type": row.get(raw_doc_type_col),
                    "project_document_type": row.get("project_document_type"),
                    "title": row.get("title"),
                    "source_label": row.get("source_label"),
                    "url": row.get("url"),
                    "source_text": row.get(text_col),
                    "matched_keyword": extract_first_keyword(row.get(text_col, "")),
                    "entity_text": entity_text,
                    "entity_key": entity_text.lower(),
                    "entity_label": entity.label_,
                    "entity_start_char": entity.start_char,
                    "entity_end_char": entity.end_char,
                }
            )

    return pd.DataFrame(records)

In [4]:
# We define a helper that creates top-entity summary tables with normalized rates.
def summarize_top_entities(
    entity_frame,
    contexts_per_type,
    documents_per_type,
    doc_id_col,
    label=None,
    top_n=10,
):
    summary_frame = entity_frame.copy()
    if label is not None:
        summary_frame = summary_frame[summary_frame["entity_label"] == label].copy()
    if summary_frame.empty:
        return pd.DataFrame()

    grouped = (
        summary_frame.groupby(["project_document_type", "entity_label", "entity_key"])
        .agg(
            mention_count=("entity_text", "size"),
            unique_contexts=("sentence_id", pd.Series.nunique),
            unique_documents=(doc_id_col, pd.Series.nunique),
            example_entity=(
                "entity_text",
                lambda series: Counter(series).most_common(1)[0][0],
            ),
        )
        .reset_index()
    )
    grouped["contexts_in_type"] = grouped["project_document_type"].map(
        contexts_per_type
    )
    grouped["documents_in_type"] = grouped["project_document_type"].map(
        documents_per_type
    )
    grouped["mentions_per_1000_contexts"] = (
        grouped["mention_count"] / grouped["contexts_in_type"] * 1000
    ).round(2)
    grouped["context_share_within_type"] = (
        grouped["unique_contexts"] / grouped["contexts_in_type"]
    ).round(4)
    grouped = grouped.sort_values(
        ["project_document_type", "mention_count", "unique_contexts"],
        ascending=[True, False, False],
    )
    return (
        grouped.groupby("project_document_type", group_keys=False)
        .head(top_n)
        .reset_index(drop=True)
    )

In [5]:
# We define a plotting helper for faceted top-entity comparisons by document type.
def plot_top_entities_by_type(
    summary_frame, plot_title, x_column="mention_count", top_n=8
):
    fig, axes = plt.subplots(2, 2, figsize=(16, 11), constrained_layout=True)
    axes = axes.flatten()

    for axis, doc_type in zip(axes, DOC_TYPE_ORDER):
        subset = (
            summary_frame[summary_frame["project_document_type"] == doc_type]
            .head(top_n)
            .copy()
        )
        if subset.empty:
            axis.text(
                0.5, 0.5, "No entities available", ha="center", va="center", fontsize=11
            )
            axis.set_axis_off()
            continue

        subset = subset.sort_values(x_column, ascending=True)
        sns.barplot(
            data=subset,
            x=x_column,
            y="example_entity",
            color=DOC_TYPE_COLORS[doc_type],
            ax=axis,
        )
        axis.set_title(doc_type if doc_type == "NGO" else doc_type.title(), fontsize=12)
        axis.set_xlabel(x_column.replace("_", " ").title())
        axis.set_ylabel("Entity")

    fig.suptitle(plot_title, fontsize=16, y=1.02)
    return fig

## 2. Load and Inspect Data

This section loads the sentence-level corpus, detects the core columns automatically, and checks the data structure before filtering to equity-related contexts.

In [6]:
# We locate the corpus, detect key columns, and load the sentence-level dataset for analysis.
PROJECT_ROOT = find_project_root(Path.cwd())
DATASET_CANDIDATES = [
    PROJECT_ROOT / "01_data" / "full_sentence_corpus.csv",
    PROJECT_ROOT / "01_data" / "validated_dataset.csv",
]
DATASET_PATH = next((path for path in DATASET_CANDIDATES if path.exists()), None)

if DATASET_PATH is None:
    raise FileNotFoundError("No expected dataset file was found in 01_data.")

analysis_df = pd.read_csv(DATASET_PATH)
TEXT_COLUMN = detect_column(
    analysis_df.columns, ["sentence_text", "text", "sentence", "context_window"]
)
DOC_ID_COLUMN = detect_column(analysis_df.columns, ["document_id", "doc_id", "id"])
RAW_DOC_TYPE_COLUMN = detect_column(
    analysis_df.columns, ["category", "document_type", "doc_type", "source_type"]
)
SENTENCE_ID_COLUMN = detect_column(
    analysis_df.columns, ["sentence_id", "context_id", "row_id"], required=False
)

analysis_df[TEXT_COLUMN] = analysis_df[TEXT_COLUMN].fillna("").astype(str)
analysis_df["project_document_type"] = analysis_df[RAW_DOC_TYPE_COLUMN].map(
    map_project_doc_type
)
analysis_df["project_document_type"] = pd.Categorical(
    analysis_df["project_document_type"],
    categories=DOC_TYPE_ORDER + ["other"],
    ordered=True,
)

preview_columns = [
    column
    for column in [
        DOC_ID_COLUMN,
        SENTENCE_ID_COLUMN,
        RAW_DOC_TYPE_COLUMN,
        TEXT_COLUMN,
        "title",
        "source_label",
    ]
    if column is not None and column in analysis_df.columns
]

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset selected: {DATASET_PATH.relative_to(PROJECT_ROOT)}")
print(f"Dataset shape: {analysis_df.shape}")
print(f"Configured text column: {TEXT_COLUMN}")
print(f"Configured document ID column: {DOC_ID_COLUMN}")
print(f"Configured raw document-type column: {RAW_DOC_TYPE_COLUMN}")
display(analysis_df[preview_columns].head(5))

Project root: /Users/far/IDS570_Text_as_Data_final_Project
Dataset selected: 01_data/full_sentence_corpus.csv
Dataset shape: (27365, 22)
Configured text column: sentence_text
Configured document ID column: document_id
Configured raw document-type column: category


,document_id,sentence_id,category,sentence_text,title,source_label
0,doc_2d830432db58,doc_2d830432db58_s00000,federal_policy,eveloping Health Equity Measures Prepared for the Office of the Assistant Secretary for Planning and Evaluation (ASP...,Developing Health Equity Measures,ASPE / HHS
1,doc_2d830432db58,doc_2d830432db58_s00001,federal_policy,"ASPE leads special initiatives; coordinates the Department's evaluation, research, and demonstration activities; and...",Developing Health Equity Measures,ASPE / HHS
2,doc_2d830432db58,doc_2d830432db58_s00002,federal_policy,"Integral to this role, ASPE conducts research and evaluation studies; develops policy analyses; and estimates the co...",Developing Health Equity Measures,ASPE / HHS
3,doc_2d830432db58,doc_2d830432db58_s00003,federal_policy,Office of Health Policy The Office of Health Policy (HP) provides a cross-cutting policy perspective that bridges De...,Developing Health Equity Measures,ASPE / HHS
4,doc_2d830432db58,doc_2d830432db58_s00004,federal_policy,"HP carries out this mission by conducting policy, economic and budget analyses, assisting in the development and rev...",Developing Health Equity Measures,ASPE / HHS


In [8]:
# We inspect data quality and show the document-type imbalance before running NER.
inspection_columns = [
    column
    for column in [
        DOC_ID_COLUMN,
        RAW_DOC_TYPE_COLUMN,
        TEXT_COLUMN,
        "title",
        "source_label",
        SENTENCE_ID_COLUMN,
    ]
    if column is not None and column in analysis_df.columns
]
missing_summary = (
    analysis_df[inspection_columns].isna().sum().rename("missing_values").reset_index()
)
missing_summary = missing_summary.rename(columns={"index": "column"})

raw_sentence_distribution = (
    analysis_df[RAW_DOC_TYPE_COLUMN]
    .value_counts(dropna=False)
    .rename("sentence_rows")
    .reset_index()
)
raw_sentence_distribution = raw_sentence_distribution.rename(
    columns={"index": "raw_document_type"}
)

project_sentence_distribution = (
    analysis_df["project_document_type"]
    .value_counts(dropna=False)
    .rename("sentence_rows")
    .reset_index()
)
project_sentence_distribution = project_sentence_distribution.rename(
    columns={"index": "project_document_type"}
)

project_document_distribution = (
    analysis_df.groupby("project_document_type", observed=False)[DOC_ID_COLUMN]
    .nunique()
    .rename("unique_documents")
    .reset_index()
    .sort_values("project_document_type")
)

imbalance_table = project_sentence_distribution.merge(
    project_document_distribution,
    on="project_document_type",
    how="left",
)
imbalance_table["sentence_share_pct"] = (
    imbalance_table["sentence_rows"] / imbalance_table["sentence_rows"].sum() * 100
).round(2)

print("Missing values in key columns:")
display(missing_summary)

print("Raw category distribution (sentence rows):")
display(raw_sentence_distribution)

print("Project document-type distribution after mapping categories:")
display(imbalance_table)

policy_note = "Policy aggregates federal_policy and state_local to align with the project framing."
dominant_type = imbalance_table.sort_values("sentence_rows", ascending=False).iloc[0][
    "project_document_type"
]
print(policy_note)
print(f"The largest document type in the sentence-level corpus is: {dominant_type}")
print(
    "This imbalance is why the notebook reports both raw counts and normalized rates per 1,000 equity-related contexts."
)

Missing values in key columns:


,column,missing_values
0,document_id,0
1,category,0
2,sentence_text,0
3,title,0
4,source_label,24
5,sentence_id,0


Raw category distribution (sentence rows):


,category,sentence_rows
0,state_local,12337
1,federal_policy,8533
2,ngo_nonprofit,5581
3,news_commentary,483
4,academic,431


Project document-type distribution after mapping categories:


,project_document_type,sentence_rows,unique_documents,sentence_share_pct
0,policy,20870,39,76.270
1,NGO,5581,17,20.390
2,news,483,23,1.770
3,academic,431,49,1.580
4,other,0,0,0.000


Policy aggregates federal_policy and state_local to align with the project framing.
The largest document type in the sentence-level corpus is: policy
This imbalance is why the notebook reports both raw counts and normalized rates per 1,000 equity-related contexts.


## 3. Filter Equity-Related Contexts

This section keeps only the sentence-level contexts that mention the target terms `equity`, `equitable`, `inequity`, or `inequitable`.

In [ ]:
# We filter the corpus to equity-related contexts and summarize what remains for NER.
equity_pattern = re.compile(
    r"\b(?:equity|equitable|inequity|inequitable)\b", flags=re.IGNORECASE
)
analysis_df["matched_terms"] = analysis_df[TEXT_COLUMN].str.findall(equity_pattern)
filtered_df = analysis_df[analysis_df["matched_terms"].map(len) > 0].copy()
filtered_df["matched_keyword"] = filtered_df["matched_terms"].map(
    lambda terms: sorted({term.lower() for term in terms})[0] if terms else None
)
filtered_df["project_document_type"] = pd.Categorical(
    filtered_df["project_document_type"],
    categories=DOC_TYPE_ORDER + ["other"],
    ordered=True,
)

keyword_distribution = (
    filtered_df["matched_keyword"]
    .value_counts(dropna=False)
    .rename("matched_rows")
    .reset_index()
)
keyword_distribution = keyword_distribution.rename(columns={"index": "matched_keyword"})

filtered_type_distribution = (
    filtered_df["project_document_type"]
    .value_counts(dropna=False)
    .rename("equity_context_rows")
    .reset_index()
)
filtered_type_distribution = filtered_type_distribution.rename(
    columns={"index": "project_document_type"}
)

match_share = len(filtered_df) / len(analysis_df) if len(analysis_df) else 0
sample_columns = [
    column
    for column in [
        DOC_ID_COLUMN,
        SENTENCE_ID_COLUMN,
        "project_document_type",
        "matched_keyword",
        TEXT_COLUMN,
    ]
    if column is not None and column in filtered_df.columns
]

print(f"Equity-related rows matched: {len(filtered_df):,}")
print(f"Share of the full corpus retained: {match_share:.2%}")
display(keyword_distribution)
display(filtered_type_distribution)

if filtered_df.empty:
    print(
        "No equity-related contexts were found, so the downstream NER sections will remain empty."
    )
else:
    print("Sample equity-related contexts:")
    display(filtered_df[sample_columns].head(8))

## 4. Run spaCy NER

This section loads a spaCy English model and extracts named entities from the equity-related contexts.

In [ ]:
# We load a spaCy English model and confirm that the NER pipeline is ready.
nlp, SPACY_MODEL_NAME = load_spacy_model()
pipe_names = ", ".join(nlp.pipe_names)

print(f"Loaded spaCy model: {SPACY_MODEL_NAME}")
print(f"Available pipeline components: {pipe_names}")
print(
    "The notebook prefers larger English models when available and falls back to en_core_web_sm when needed."
)

In [ ]:
# We run spaCy NER on the filtered contexts and build a long-format entity dataset.
entity_df = build_entity_dataset(
    filtered_df, TEXT_COLUMN, DOC_ID_COLUMN, RAW_DOC_TYPE_COLUMN, nlp
)

if entity_df.empty:
    print("No entities were extracted from the filtered corpus.")
else:
    entity_df["project_document_type"] = pd.Categorical(
        entity_df["project_document_type"],
        categories=DOC_TYPE_ORDER + ["other"],
        ordered=True,
    )
    entity_df["entity_label"] = entity_df["entity_label"].astype(str)
    contexts_with_entities = (
        entity_df["sentence_id"].nunique()
        if "sentence_id" in entity_df.columns
        else len(filtered_df)
    )
    unique_entities = entity_df["entity_key"].nunique()

    print(f"Filtered contexts processed: {len(filtered_df):,}")
    print(f"Entity mentions extracted: {len(entity_df):,}")
    print(f"Unique normalized entities: {unique_entities:,}")
    print(f"Contexts with at least one entity: {contexts_with_entities:,}")

    preview_columns = [
        column
        for column in [
            "project_document_type",
            "matched_keyword",
            "entity_text",
            "entity_label",
            "title",
        ]
        if column in entity_df.columns
    ]
    display(entity_df[preview_columns].head(12))

## 5. Build Entity-Level Dataset

The entity-level table below is the core long-format dataset used for the rest of the analysis. Each row represents one named entity mention found inside an equity-related sentence.

In [ ]:
# We summarize all entity labels first and then isolate the ORG, GPE, and PERSON subsets.
if entity_df.empty:
    overall_label_freq = pd.DataFrame(columns=["entity_label", "mentions", "share_pct"])
    label_by_type_counts = pd.DataFrame()
    label_by_type_props = pd.DataFrame()
    focus_entity_df = pd.DataFrame()
    print("Entity summaries are empty because no named entities were extracted.")
else:
    overall_label_freq = (
        entity_df["entity_label"].value_counts().rename("mentions").reset_index()
    )
    overall_label_freq = overall_label_freq.rename(columns={"index": "entity_label"})
    overall_label_freq["share_pct"] = (
        overall_label_freq["mentions"] / overall_label_freq["mentions"].sum() * 100
    ).round(2)

    label_order = overall_label_freq["entity_label"].tolist()
    label_by_type_counts = pd.crosstab(
        entity_df["project_document_type"], entity_df["entity_label"]
    ).reindex(DOC_TYPE_ORDER, fill_value=0)
    label_by_type_counts = label_by_type_counts.reindex(
        columns=label_order, fill_value=0
    )

    label_by_type_props = pd.crosstab(
        entity_df["project_document_type"],
        entity_df["entity_label"],
        normalize="index",
    ).reindex(DOC_TYPE_ORDER, fill_value=0)
    label_by_type_props = label_by_type_props.reindex(
        columns=label_order, fill_value=0
    ).round(3)

    focus_entity_df = entity_df[entity_df["entity_label"].isin(FOCUS_LABELS)].copy()
    focus_counts = pd.crosstab(
        focus_entity_df["project_document_type"], focus_entity_df["entity_label"]
    ).reindex(DOC_TYPE_ORDER, fill_value=0)

    print("Overall entity label frequencies:")
    display(overall_label_freq.head(15))

    print("Entity label counts by document type:")
    display(label_by_type_counts.iloc[:, :10])

    print("Entity label proportions within each document type:")
    display(label_by_type_props.iloc[:, :10])

    print("Focused counts for ORG, GPE, and PERSON:")
    display(focus_counts)

## 6. Aggregate and Compare Results

These tables compare overall entity activity and the top co-occurring entities across academic, policy, NGO, and news contexts.

In [ ]:
# We compare document types with both raw entity counts and imbalance-adjusted rates.
if entity_df.empty or filtered_df.empty:
    contexts_per_type = pd.Series(dtype=float)
    documents_per_type = pd.Series(dtype=float)
    entity_summary_by_type = pd.DataFrame()
    overall_top_entities = pd.DataFrame()
    print(
        "Comparison tables are empty because the filtered or entity dataset is empty."
    )
else:
    contexts_per_type = (
        filtered_df.groupby("project_document_type", observed=False)
        .size()
        .reindex(DOC_TYPE_ORDER, fill_value=0)
    )
    documents_per_type = (
        filtered_df.groupby("project_document_type", observed=False)[DOC_ID_COLUMN]
        .nunique()
        .reindex(DOC_TYPE_ORDER, fill_value=0)
    )
    entity_mentions_by_type = (
        entity_df.groupby("project_document_type", observed=False)
        .size()
        .reindex(DOC_TYPE_ORDER, fill_value=0)
    )

    entity_summary_by_type = pd.DataFrame(
        {
            "equity_contexts": contexts_per_type,
            "unique_documents": documents_per_type,
            "entity_mentions": entity_mentions_by_type,
        }
    )
    entity_summary_by_type["mentions_per_1000_contexts"] = (
        entity_summary_by_type["entity_mentions"]
        / entity_summary_by_type["equity_contexts"]
        * 1000
    ).round(2)
    entity_summary_by_type["share_of_all_entity_mentions_pct"] = (
        entity_summary_by_type["entity_mentions"]
        / entity_summary_by_type["entity_mentions"].sum()
        * 100
    ).round(2)

    overall_top_entities = summarize_top_entities(
        entity_df,
        contexts_per_type.to_dict(),
        documents_per_type.to_dict(),
        DOC_ID_COLUMN,
        label=None,
        top_n=10,
    )

    print("Entity counts by document type:")
    display(
        entity_summary_by_type.reset_index().rename(
            columns={"index": "project_document_type"}
        )
    )

    print("Top co-occurring entities in each document type:")
    display(
        overall_top_entities[
            [
                "project_document_type",
                "example_entity",
                "entity_label",
                "mention_count",
                "unique_contexts",
                "mentions_per_1000_contexts",
            ]
        ]
    )

    print(
        "Normalized rates are especially important here because policy contexts greatly outnumber the other document types."
    )

In [ ]:
# We create clean summary tables for the top organizations, places, and people in each document type.
if entity_df.empty or filtered_df.empty:
    top_org = pd.DataFrame()
    top_gpe = pd.DataFrame()
    top_person = pd.DataFrame()
    print("Top-entity tables are empty because the entity dataset is empty.")
else:
    top_org = summarize_top_entities(
        entity_df,
        contexts_per_type.to_dict(),
        documents_per_type.to_dict(),
        DOC_ID_COLUMN,
        label="ORG",
        top_n=10,
    )
    top_gpe = summarize_top_entities(
        entity_df,
        contexts_per_type.to_dict(),
        documents_per_type.to_dict(),
        DOC_ID_COLUMN,
        label="GPE",
        top_n=10,
    )
    top_person = summarize_top_entities(
        entity_df,
        contexts_per_type.to_dict(),
        documents_per_type.to_dict(),
        DOC_ID_COLUMN,
        label="PERSON",
        top_n=10,
    )

    top_columns = [
        "project_document_type",
        "example_entity",
        "mention_count",
        "unique_contexts",
        "unique_documents",
        "mentions_per_1000_contexts",
    ]

    print("Top 10 ORG entities by document type:")
    display(top_org[top_columns])

    print("Top 10 GPE entities by document type:")
    display(top_gpe[top_columns])

    print("Top 10 PERSON entities by document type:")
    display(top_person[top_columns])

    print(
        "The PERSON table is usually the noisiest because sentence-level NER can misclassify programs or acronyms as people in technical policy prose."
    )

## 7. Visualizations

These figures translate the tables above into presentation-ready charts for the class project.

In [ ]:
# We visualize the overall entity label distribution across all equity-related contexts.
if overall_label_freq.empty:
    print("No label-distribution plot was created because the entity dataset is empty.")
else:
    label_plot_df = overall_label_freq.head(12).copy()
    plt.figure(figsize=(10, 6))
    sns.barplot(data=label_plot_df, x="mentions", y="entity_label", color="#1f4e79")
    plt.title("Overall Entity Label Distribution in Equity-Related Contexts")
    plt.xlabel("Entity mentions")
    plt.ylabel("Entity label")
    plt.tight_layout()
    plt.show()
    print(f"Displayed the top {len(label_plot_df)} entity labels by raw mention count.")

In [ ]:
# We compare document types with side-by-side raw and normalized entity-count charts.
if entity_summary_by_type.empty:
    print(
        "No document-type comparison plot was created because the summary table is empty."
    )
else:
    plot_summary = entity_summary_by_type.reset_index(
        names="project_document_type"
    ).copy()
    plot_summary = plot_summary[
        plot_summary["project_document_type"].isin(DOC_TYPE_ORDER)
    ]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
    sns.barplot(
        data=plot_summary,
        x="project_document_type",
        y="entity_mentions",
        hue="project_document_type",
        palette=DOC_TYPE_COLORS,
        legend=False,
        ax=axes[0],
    )
    axes[0].set_title("Raw Entity Mentions by Document Type")
    axes[0].set_xlabel("Document type")
    axes[0].set_ylabel("Entity mentions")

    sns.barplot(
        data=plot_summary,
        x="project_document_type",
        y="mentions_per_1000_contexts",
        hue="project_document_type",
        palette=DOC_TYPE_COLORS,
        legend=False,
        ax=axes[1],
    )
    axes[1].set_title("Entity Mentions per 1,000 Equity Contexts")
    axes[1].set_xlabel("Document type")
    axes[1].set_ylabel("Mentions per 1,000 contexts")

    plt.show()
    print(
        "The left panel shows raw counts, while the right panel adjusts for the strong policy-heavy imbalance in the corpus."
    )

In [ ]:
# We plot the most frequent organizations within each document type.
if top_org.empty:
    print("No ORG plot was created because the ORG summary table is empty.")
else:
    org_figure = plot_top_entities_by_type(
        top_org,
        plot_title="Top ORG Entities Near Equity-Related Contexts",
        x_column="mention_count",
        top_n=8,
    )
    plt.show()
    print(
        "Each panel shows the leading organizations within one document type using raw mention counts."
    )

In [ ]:
# We plot the most frequent places within each document type.
if top_gpe.empty:
    print("No GPE plot was created because the GPE summary table is empty.")
else:
    gpe_figure = plot_top_entities_by_type(
        top_gpe,
        plot_title="Top GPE Entities Near Equity-Related Contexts",
        x_column="mention_count",
        top_n=8,
    )
    plt.show()
    print(
        "These panels highlight the main geographic references attached to equity-related language."
    )

In [ ]:
# We plot the most frequent person entities within each document type.
if top_person.empty:
    print("No PERSON plot was created because the PERSON summary table is empty.")
else:
    person_figure = plot_top_entities_by_type(
        top_person,
        plot_title="Top PERSON Entities Near Equity-Related Contexts",
        x_column="mention_count",
        top_n=8,
    )
    plt.show()
    print(
        "PERSON results should be interpreted cautiously because policy text often produces noisier person labels than ORG or GPE."
    )

## 8. Interpretation and Summary

The final section turns the tables and figures into a short analytical summary for the project write-up.

In [ ]:
# We translate the main tables into a short interpretation that accounts for class imbalance.
if entity_df.empty or entity_summary_by_type.empty:
    print("No interpretation was generated because the analysis tables are empty.")
else:
    focal_label_totals = overall_label_freq[
        overall_label_freq["entity_label"].isin(FOCUS_LABELS)
    ].copy()
    most_common_focal_label = (
        focal_label_totals.iloc[0]["entity_label"]
        if not focal_label_totals.empty
        else "ORG"
    )
    raw_leader = entity_summary_by_type["entity_mentions"].idxmax()
    normalized_leader = entity_summary_by_type["mentions_per_1000_contexts"].idxmax()

    def top_entity_name(summary_frame, doc_type):
        subset = summary_frame[summary_frame["project_document_type"] == doc_type]
        if subset.empty:
            return None
        return subset.iloc[0]["example_entity"]

    policy_org = top_entity_name(top_org, "policy")
    ngo_org = top_entity_name(top_org, "NGO")
    news_person = top_entity_name(top_person, "news")
    policy_gpe = top_entity_name(top_gpe, "policy")

    interpretation_lines = [
        f"The most common focal entity label is {most_common_focal_label}, which means organizations dominate the equity-related named-entity landscape.",
        f"Raw entity totals are highest in {raw_leader} texts, but this partly reflects corpus imbalance rather than only stronger entity density.",
        f"After normalizing by equity-related contexts, the highest entity rate appears in {normalized_leader} texts, which gives a fairer cross-category comparison.",
    ]

    if policy_org:
        interpretation_lines.append(
            f"Policy texts most often mention institutions such as {policy_org}, which fits the policy corpus's emphasis on agencies, public programs, and administrative actors."
        )
    if ngo_org:
        interpretation_lines.append(
            f"NGO texts foreground organizations such as {ngo_org}, suggesting a stronger emphasis on advocacy groups, coalitions, and nonprofit actors."
        )
    if policy_gpe:
        interpretation_lines.append(
            f"Place references in policy writing are led by entities such as {policy_gpe}, which indicates that policy discussions often connect equity to states, counties, and implementation sites."
        )
    if news_person:
        interpretation_lines.append(
            f"News texts include more visible person-level references such as {news_person}, although the news subset is small and the PERSON label is noisier than ORG or GPE."
        )

    interpretation_lines.append(
        "Because policy contexts substantially outnumber the other categories, the normalized rates and within-type proportions are more informative than raw counts alone when comparing document types."
    )
    interpretation_lines.append(
        "A final caution is that spaCy occasionally misclassifies programs, acronyms, or initiative names as people, so PERSON findings should be treated as suggestive rather than definitive."
    )

    print("Key interpretation points:")
    for line in interpretation_lines:
        print(f"- {line}")